In [ ]:
# %pip install anthropic

# curl https://api.anthropic.com/v1/messages \
#         --header "x-api-key: sk-ant-..." \
#         --header "anthropic-version: 2023-06-01" \
#         --header "content-type: application/json" \
#         --data \
#     '{
#         "model": "claude-sonnet-4-6",
#         "max_tokens": 1024,
#         "messages": [
#             {"role": "user", "content": "Hello, world"}
#         ]
#     }'

# {
#     "model": "claude-sonnet-4-6",
#     "id": "msg_01X4FZsFyNT9UaYHvJJhjkjb",
#     "type": "message",
#     "role": "assistant",
#     "content": [
#         {
#             "type": "text",
#             "text": "Hello! 👋 How are you doing? Is there something I can help you with today?"
#         }
#     ],
#     "stop_reason": "end_turn",
#     "stop_sequence": null,
#     "stop_details": null,
#     "usage": {"input_jannijannijannijannijannijannijannijannijannijannijannikkuhljannijannikkuhljannijannijannijannijannijannijannijannijannijannijannijannijannijannijannijannijanni

# Setup

In [ ]:
import os
import time
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

import base64
import json

load_dotenv(find_dotenv())

client = Anthropic()

In [14]:
# claude-opus-4-7
# claude-sonnet-4-6
# claude-haiku-4-5

# Count tokens in a Message
[Claude API Docs](https://platform.claude.com/docs/en/api/python/messages/count_tokens)

In [ ]:
Opus4_7_message_tokens_count = client.messages.count_tokens(
    messages=[{
        "content": "Hello, world",
        "role": "user",
    }],
    model="claude-opus-4-7",
)

Opus4_6_message_tokens_count = client.messages.count_tokens(
    messages=[{
        "content": "Hello, world",
        "role": "user",
    }],
    model="claude-opus-4-6",
)

Sonnet4_6_message_tokens_count = client.messages.count_tokens(
    messages=[{
        "content": "Hello, world",
        "role": "user",
    }],
    model="claude-sonnet-4-6",
)

Haiku4_5_message_tokens_count = client.messages.count_tokens(
    messages=[{
        "content": "Hello, world",
        "role": "user",
    }],
    model="claude-haiku-4-5",
)

print("Opus 4.7:", Opus4_7_message_tokens_count.input_tokens)
print("Opus 4.6:", Opus4_6_message_tokens_count.input_tokens)
print("Sonnet 4.6:", Sonnet4_6_message_tokens_count.input_tokens)
print("Haiku 4.5:", Haiku4_5_message_tokens_count.input_tokens)

# PDF Upload via API

In [ ]:
uploaded = client.beta.files.upload(
    file=("varta ag_2021_report.pdf", open("../../localdata/esg_reports/Baseline A remaining/varta ag_2021_report.pdf", "rb"), "application/pdf"),
)
documentID = uploaded.id
documentID


In [15]:
documentID = "file_011CbPQ8APW12KzZwRPm2RAY"

In [16]:
with open("../../baselines/baseline_a_frontier_model/BaselineA-Promt.txt", "r", encoding="utf-8") as f:
    prompt_text = f.read()

prompt_text

'You are an expert in corporate sustainability reporting and greenhouse gas accounting.\n\nExtract all greenhouse gas emissions data from the provided sustainability report.\nReport values exactly as stated in the document. Do not convert or normalize units.\nDo not include years for which no data is reported.\nIf multiple values are reported for the same scope and year, prefer the total/consolidated value. If no total/consolidated value is identifiable, include all values as separate entries in the array and label them.\n\nReturn your answer as a JSON object with exactly this structure:\n\n{\n  "report_name": "<filename without .pdf>",\n  "report_title": "<full report title>",\n  "emissions": {\n    "scope_1": {\n      "<year>": [{"value": <number>, "unit": "<string>", "label": "<string>"}]\n    },\n    "scope_2_market_based": {\n      "<year>": [{"value": <number>, "unit": "<string>", "label": "<string>"}]\n    },\n    "scope_2_location_based": {\n      "<year>": [{"value": <number>,

In [17]:
MODEL_USE = "claude-sonnet-4-6"

document_batch = client.beta.messages.batches.create(
    betas=["files-api-2025-04-14"],
    requests=[{
        "custom_id": "docuTest",
        "params": {
            "max_tokens": 4096,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text},
                    {
                        "type": "document",
                        "source": {
                            "type": "file",
                            "file_id": documentID,
                        },
                    },
                ]
            }],
            "model": MODEL_USE,
        },
    }],
)
DOCUMENTMESSAGE_BATCH_ID = document_batch.id
print(document_batch)

BetaMessageBatch(id='msgbatch_019HJRmpXw1HjYFDJx6fH3XB', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 5, 25, 15, 10, 15, 42378, tzinfo=datetime.timezone.utc), ended_at=None, expires_at=datetime.datetime(2026, 5, 26, 15, 10, 15, 42378, tzinfo=datetime.timezone.utc), processing_status='in_progress', request_counts=BetaMessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=1, succeeded=0), results_url=None, type='message_batch')


# Claude API Batch

In [18]:
# message_batch = client.messages.batches.create(
#     requests=[{
#         "custom_id": "testung-2",
#         "params": {
#             "max_tokens": 1024,
#             "messages": [{
#                 "content": "Hello, world",
#                 "role": "user",
#             }],
#             "model": "claude-haiku-4-5",
#         },
#     }],
# )
# MESSAGE_BATCH_ID = message_batch.id
# print(MESSAGE_BATCH_ID)

In [19]:
if DOCUMENTMESSAGE_BATCH_ID != 0:
    MESSAGE_BATCH_ID = DOCUMENTMESSAGE_BATCH_ID

message_batch = None
while True:
    message_batch = client.messages.batches.retrieve(MESSAGE_BATCH_ID)
    if message_batch.processing_status == "ended":
        break

    print(f"Batch {MESSAGE_BATCH_ID} is still processing...")
    time.sleep(20)
print(message_batch)

Batch msgbatch_019HJRmpXw1HjYFDJx6fH3XB is still processing...
Batch msgbatch_019HJRmpXw1HjYFDJx6fH3XB is still processing...
MessageBatch(id='msgbatch_019HJRmpXw1HjYFDJx6fH3XB', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 5, 25, 15, 10, 15, 42378, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 5, 25, 15, 11, 13, 411162, tzinfo=TzInfo(0)), expires_at=datetime.datetime(2026, 5, 26, 15, 10, 15, 42378, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=1), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_019HJRmpXw1HjYFDJx6fH3XB/results', type='message_batch')


In [32]:
# Die for-Schleife ruft jedes mal im Hintergrund '__next__()' auf und nur so wird aus einem Pointer der Inhalt
for batch in client.messages.batches.results(
    MESSAGE_BATCH_ID,
):
    with open("result.json", "w", encoding="utf-8") as f:
        f.write(batch.model_dump_json(indent=2))

In [ ]:
match batch.result.message.model:
    case "claude-haiku-4-5":
        PREIS_PRO_1M_INPUT_USD = 0.59
        PREIS_PRO_1M_OUTPUT_USD = 2.97
    case "claude-sonnet-4-6":
        PREIS_PRO_1M_INPUT_USD = 1.78
        PREIS_PRO_1M_OUTPUT_USD = 8.92
    case "claude-opus-4-7":
        PREIS_PRO_1M_INPUT_USD = 2.97
        PREIS_PRO_1M_OUTPUT_USD = 14.88

gesamtkosten_batch = 0.0

for item in client.messages.batches.results(MESSAGE_BATCH_ID):
    
    # --- 2. Token direkt aus dem Objekt auslesen ---
    in_tokens = item.result.message.usage.input_tokens
    out_tokens = item.result.message.usage.output_tokens
        
    # --- 3. Kosten für DIESEN Report berechnen ---
    kosten_input  = (in_tokens  / 1_000_000) * PREIS_PRO_1M_INPUT_USD
    kosten_output = (out_tokens / 1_000_000) * PREIS_PRO_1M_OUTPUT_USD
    
    kosten_report = kosten_input + kosten_output
    gesamtkosten_batch += kosten_report
    
    raw_text = item.result.message.content[0].text
    
    clean_text = raw_text.strip()
    if clean_text.startswith("```json"):
        # Schneidet "```json\n" vorne und "\n```" hinten ab
        clean_text = clean_text[7:-3].strip() 
    elif clean_text.startswith("```"):
        clean_text = clean_text[3:-3].strip()
        
    with open("output.json", "w", encoding="utf-8") as f:
                json.dump(json.loads(clean_text), f, indent=2, ensure_ascii=False)
                
    print(f"   Tokens: {in_tokens} In | {out_tokens} Out")
    print(f"   Kosten: ${kosten_report:.4f}\n")
                
print("-" * 30)
print(f"GESAMTKOSTEN FÜR DIESEN BATCH: ${gesamtkosten_batch:.4f}")
print(f"Mit Model: {MODEL_USE}")
print("-" * 30)


   Tokens: 84061 In | 449 Out
   Kosten: $0.1536

------------------------------
GESAMTKOSTEN FÜR DIESEN BATCH: $0.1536
Mit Model: $claude-sonnet-4-6
------------------------------


In [36]:
client.beta.files.delete(file_id=documentID,)

DeletedFile(id='file_011CbPQ8APW12KzZwRPm2RAY', type='file_deleted')